# Basic Models — Sequence-to-Sequence Architectures

## 1. Sequence-to-Sequence Architecture

A **Sequence-to-Sequence (Seq2Seq)** model maps an input into an output sequence. The two sequences do not need to have the same length.

For example, in machine translation:

```text
Input:
Jane visite l'Afrique en septembre

Output:
Jane is visiting Africa in September
```

The input contains $T_x$ tokens, while the output contains $T_y$ tokens:

$$
T_x \neq T_y
$$

The general idea is:

```mermaid
flowchart LR
    A[Input] --> B[Encoder]
    B --> C[Representation]
    C --> D[Decoder]
    D --> E[Output Sequence]
```

The encoder processes the input and produces a representation. The decoder uses that representation to generate the output sequence one token at a time.

The core abstraction is:

$$
\boxed{
\text{Input}
\rightarrow
\text{Encoder}
\rightarrow
\text{Representation}
\rightarrow
\text{Decoder}
\rightarrow
\text{Output Sequence}
}
$$

A useful example of this basic architecture is the following sequence-to-sequence diagram from *Dive into Deep Learning*:

![Sequence-to-Sequence architecture](https://d2l.ai/_images/seq2seq.svg)

The diagram illustrates an RNN encoder processing the source sequence and an RNN decoder generating the target sequence token by token. The encoder's final hidden state is passed to the decoder, while special beginning-of-sequence and end-of-sequence tokens control generation.

---

## 2. Machine Translation

Machine translation is the canonical example of a Seq2Seq problem.

Suppose the source sentence is:

```text
Jane visite l'Afrique en septembre
```

and the desired translation is:

```text
Jane is visiting Africa in September
```

The input sequence is:

$$
x =
\left[
x^{\langle 1\rangle},
x^{\langle 2\rangle},
\ldots,
x^{\langle T_x\rangle}
\right]
$$

The target sequence is:

$$
y =
\left[
y^{\langle 1\rangle},
y^{\langle 2\rangle},
\ldots,
y^{\langle T_y\rangle}
\right]
$$

The encoder processes the source sequence:

$$
x^{\langle 1\rangle}
\rightarrow
x^{\langle 2\rangle}
\rightarrow
\cdots
\rightarrow
x^{\langle T_x\rangle}
$$

and produces hidden states:

$$
a^{\langle 1\rangle},
a^{\langle 2\rangle},
\ldots,
a^{\langle T_x\rangle}
$$

In the basic architecture, the final encoder state is used as the representation passed to the decoder:

$$
c = a^{\langle T_x\rangle}
$$

Therefore:

$$
\boxed{
x_1,\ldots,x_{T_x}
\rightarrow
\text{Encoder}
\rightarrow
c
\rightarrow
\text{Decoder}
\rightarrow
y_1,\ldots,y_{T_y}
}
$$

The important point is that translation is **not** a one-to-one mapping between input and output tokens.

For example:

```text
Input:
I love machine learning

Output:
J'aime l'apprentissage automatique
```

There is no requirement that:

$$
x_i \leftrightarrow y_i
$$

The model learns a mapping between the **entire source sequence** and the **entire target sequence**.

---

## 3. Encoder

The encoder converts the input sequence into an internal representation.

For a recurrent encoder:

$$
a^{\langle t\rangle}
=
f
\left(
a^{\langle t-1\rangle},
x^{\langle t\rangle}
\right)
$$

At each timestep, the encoder combines:

- the current input;
- the previous hidden state.

After the complete source sequence has been processed:

$$
a^{\langle T_x\rangle}
$$

contains information accumulated from the input sequence.

In the basic Seq2Seq architecture:

$$
c = a^{\langle T_x\rangle}
$$

where $c$ is the representation passed to the decoder.

Therefore:

$$
\boxed{
\text{Encoder}
=
\text{Convert the input into a representation}
}
$$

The encoder does not generate the target sentence. Its role is to provide the information that the decoder needs.

---

## 4. Decoder and Autoregressive Generation

The decoder uses the encoded representation to generate the output sequence one token at a time.

Suppose the decoder has already generated:

$$
y^{\langle 1\rangle},
\ldots,
y^{\langle t-1\rangle}
$$

It predicts the next token:

$$
y^{\langle t\rangle}
$$

The decoder state can be represented conceptually as:

$$
s^{\langle t\rangle}
=
f
\left(
s^{\langle t-1\rangle},
y^{\langle t-1\rangle},
c
\right)
$$

The model then produces a probability distribution over the vocabulary:

$$
P
\left(
y^{\langle t\rangle}
\mid
x,
y^{\langle 1:t-1\rangle}
\right)
$$

The decoder is therefore **autoregressive**: each new prediction depends on the previously generated tokens.

```text
<SOS>
  ↓
y₁
  ↓
y₂
  ↓
y₃
  ↓
...
  ↓
<EOS>
```

Two special tokens are commonly used:

- `<SOS>` or `<BOS>`: beginning of sequence;
- `<EOS>`: end of sequence.

Generation stops when the decoder produces `<EOS>`.

This means the decoder does not need to know $T_y$ beforehand.

---

## 5. Seq2Seq as a Conditional Language Model

A key insight is that the decoder can be viewed as a **language model conditioned on the input sequence**.

A normal language model models:

$$
P(y)
$$

whereas a Seq2Seq model models:

$$
P(y \mid x)
$$

because the output depends on the source input.

The probability of an entire target sequence can be factorized as:

$$
P(y\mid x)
=
\prod_{t=1}^{T_y}
P
\left(
y^{\langle t\rangle}
\mid
x,
y^{\langle 1:t-1\rangle}
\right)
$$

This means that the decoder performs the following process:

$$
\boxed{
\text{Input Sequence}
+
\text{Previous Output Tokens}
\rightarrow
\text{Next Token Distribution}
}
$$

For example:

```text
Input:
Jane visite l'Afrique en septembre

Generated so far:
<SOS>

Possible next tokens:
Jane
She
The
...
```

After generating `Jane`, the next prediction is conditioned on:

```text
<SOS> Jane
```

and so on.

The decoder is therefore essentially a **conditional autoregressive language model**.

---

## 6. Training

During training, the correct target sequence is known.

Suppose the target is:

```text
<SOS> Jane is visiting Africa <EOS>
```

The decoder is trained to predict the next correct token:

```text
Decoder input        Target

<SOS>          ->    Jane
Jane           ->    is
is             ->    visiting
visiting       ->    Africa
Africa         ->    <EOS>
```

This training strategy is commonly called **teacher forcing**.

At each timestep, the loss encourages the correct token to receive high probability:

$$
\mathcal{L}^{\langle t\rangle}
=
-
\log
P
\left(
y^{\langle t\rangle}
\mid
x,
y^{\langle 1:t-1\rangle}
\right)
$$

The total sequence loss is:

$$
\mathcal{L}
=
-
\sum_{t=1}^{T_y}
\log
P
\left(
y^{\langle t\rangle}
\mid
x,
y^{\langle 1:t-1\rangle}
\right)
$$

The important distinction is:

### During training

The decoder typically receives the correct previous target token.

### During inference

The decoder must use its own previous prediction.

Therefore, inference follows:

$$
\hat{y}^{\langle t\rangle}
\rightarrow
\text{input for the next step}
$$

rather than using the ground-truth target.

---

## 7. Image Captioning: A Different Seq2Seq Architecture

Seq2Seq is not limited to text-to-text problems.

A second important example is **image captioning**, where the input is an image and the output is a natural-language sequence.

For example:

```text
Image:
A cat sitting on a chair

Output:
"A cat is sitting on a chair."
```

The architecture can be:

```mermaid
flowchart LR
    A[Image] --> B[CNN Encoder]
    B --> C[Image Representation]
    C --> D[RNN Decoder]
    D --> E[Caption]
```

The CNN acts as the encoder:

$$
x_{\text{image}}
\rightarrow
z_{\text{image}}
$$

where $z_{\text{image}}$ is a learned visual representation.

The decoder then generates the caption:

$$
z_{\text{image}}
\rightarrow
y^{\langle 1\rangle},
y^{\langle 2\rangle},
\ldots,
y^{\langle T_y\rangle}
$$

Conceptually:

```text
Image
  ↓
CNN Encoder
  ↓
Image Representation
  ↓
RNN Decoder
  ↓
"A"
  ↓
"cat"
  ↓
"is"
  ↓
"sitting"
  ↓
...
```

The important insight is that the **encoder and decoder solve different problems**:

$$
\boxed{
\text{Encoder}
=
\text{Understand / Encode the Input}
}
$$

$$
\boxed{
\text{Decoder}
=
\text{Generate the Output Sequence}
}
$$

The encoder does not have to be an RNN. Depending on the task, it can be a CNN or another architecture capable of producing a useful representation.

This gives the more general Seq2Seq pattern:

$$
\boxed{
\text{Input}
\rightarrow
\text{Encoder}
\rightarrow
\text{Representation}
\rightarrow
\text{Decoder}
\rightarrow
\text{Sequence}
}
$$

---

## 8. The Fundamental Limitation of Basic Seq2Seq

The basic architecture has an important bottleneck.

The entire input sequence:

$$
x^{\langle 1\rangle},
x^{\langle 2\rangle},
\ldots,
x^{\langle T_x\rangle}
$$

must be compressed into a single representation:

$$
c
$$

The decoder then uses that representation to generate the entire target sequence.

Conceptually:

```text
Long Input Sequence
        |
        v
     Encoder
        |
        v
  ONE Fixed Representation
        |
        v
     Decoder
        |
        v
 Output Sequence
```

For a short input, this may work reasonably well.

For a long input, however, the encoder must preserve a large amount of information inside a fixed-size representation.

This creates an **information bottleneck**:

$$
\boxed{
\text{Long Input}
\rightarrow
\text{Fixed Context}
\rightarrow
\text{Decoder}
}
$$

The longer and more complex the input, the more difficult it becomes for a single fixed representation to preserve every piece of information needed later by the decoder.

This limitation is the main motivation for the next major concept in Week 3:

$$
\boxed{
\text{Basic Seq2Seq}
\rightarrow
\text{Fixed-Context Bottleneck}
\rightarrow
\text{Attention}
}
$$

---

# Final Mental Model

The core idea of Basic Seq2Seq is:

$$
\boxed{
\text{Input}
\rightarrow
\text{Encoder}
\rightarrow
\text{Representation}
\rightarrow
\text{Autoregressive Decoder}
\rightarrow
\text{Output Sequence}
}
$$

For machine translation:

$$
\boxed{
\text{Source Sentence}
\rightarrow
\text{Encoder}
\rightarrow
\text{Context}
\rightarrow
\text{Decoder}
\rightarrow
\text{Target Sentence}
}
$$

For image captioning:

$$
\boxed{
\text{Image}
\rightarrow
\text{CNN Encoder}
\rightarrow
\text{Visual Representation}
\rightarrow
\text{RNN Decoder}
\rightarrow
\text{Caption}
}
$$

The decoder can be understood as a conditional language model:

$$
P(y\mid x)
$$

with autoregressive factorization:

$$
P(y\mid x)
=
\prod_{t=1}^{T_y}
P
\left(
y_t
\mid
x,
y_{<t}
\right)
$$

The central limitation is:

$$
\boxed{
\text{Entire Input}
\rightarrow
\text{One Fixed Representation}
\rightarrow
\text{Decoder}
}
$$

This fixed-context bottleneck motivates Attention.

The conceptual progression is therefore:

$$
\boxed{
\text{Basic Seq2Seq}
\rightarrow
\text{Conditional Generation}
\rightarrow
\text{Fixed-Context Bottleneck}
\rightarrow
\text{Attention}
}
$$

The most important idea to remember is:

> **Seq2Seq is a general encode → condition → generate framework. The encoder converts the input into a representation, while the decoder acts as an autoregressive conditional language model that generates the output sequence one token at a time.**

# Picking the Most Likely Sentence

In a sequence-to-sequence model, once the model has learned how to predict the next token, we need to determine **which complete output sentence should be generated**.

Given an input sequence $x$, the objective is:

$$
\boxed{
y^*
=
\arg\max_y P(y\mid x)
}
$$

That is:

> **Among all possible output sequences, choose the one with the highest probability given the input.**

---

# 1. Probability of an Entire Sentence

Suppose the output sequence is:

$$
y=
\left(
y^{\langle1\rangle},
y^{\langle2\rangle},
\ldots,
y^{\langle T_y\rangle}
\right)
$$

The probability of the complete sequence is decomposed using the chain rule:

$$
\boxed{
P(y\mid x)
=
\prod_{t=1}^{T_y}
P
\left(
y^{\langle t\rangle}
\mid
y^{\langle1\rangle},
\ldots,
y^{\langle t-1\rangle},
x
\right)
}
$$

or equivalently:

$$
\boxed{
P(y\mid x)
=
\prod_{t=1}^{T_y}
P
\left(
y^{\langle t\rangle}
\mid
y^{\langle<t\rangle},
x
\right)
}
$$

This means that the model generates the sentence **one token at a time**, and the probability of the entire sentence is the product of the conditional probabilities of its individual tokens.

For example:

> I love this movie.

has:

$$
P(y\mid x)
=
P(\text{I}\mid x)
\cdot
P(\text{love}\mid\text{I},x)
\cdot
P(\text{this}\mid\text{I love},x)
\cdot
P(\text{movie}\mid\text{I love this},x)
$$

Suppose these probabilities are:

$$
0.8,\quad
0.7,\quad
0.6,\quad
0.9
$$

Then:

$$
P(y\mid x)
=
0.8\times0.7\times0.6\times0.9
=
0.3024
$$

Therefore, the model assigns probability to the **whole sentence through the sequence of next-token predictions**.

---

# 2. Why Is Choosing the Best Sentence Difficult?

Our actual objective is:

$$
\boxed{
y^*
=
\arg\max_y P(y\mid x)
}
$$

However, at every timestep the model has many possible next tokens.

If the vocabulary size is:

$$
V
$$

then the first timestep has approximately $V$ possible choices.

Each of those choices creates another $V$ possibilities at the next timestep.

Thus, for a sequence of length $T_y$, the number of possible sequences grows roughly as:

$$
\boxed{
V^{T_y}
}
$$

For example, if:

$$
V=10000
$$

and:

$$
T_y=10
$$

then the space of possible sequences is enormous.

Therefore, we cannot simply enumerate every possible sentence, calculate its probability, and select the maximum.

The problem becomes a **search problem**:

$$
\boxed{
\text{Search the enormous output space for a high-probability sequence}
}
$$

---

# 3. Greedy Search

The simplest strategy is **Greedy Search**.

At every timestep, choose the token with the highest probability:

$$
\boxed{
y^{\langle t\rangle}
=
\arg\max_w
P
\left(
w
\mid
y^{\langle<t\rangle},
x
\right)
}
$$

For example, at the first timestep:

$$
P(\text{I}\mid x)=0.4
$$

$$
P(\text{We}\mid x)=0.3
$$

$$
P(\text{You}\mid x)=0.2
$$

Greedy Search chooses:

$$
\boxed{
y^{\langle1\rangle}=\text{I}
}
$$

The model then uses `I` as part of the generated context and predicts the next token.

The process is:

$$
\boxed{
\text{Choose best token}
\rightarrow
\text{Append it}
\rightarrow
\text{Predict next token}
\rightarrow
\text{Repeat}
}
$$

Greedy Search is simple and computationally efficient.

However, it has an important limitation.

---

# 4. Why Greedy Search Does Not Guarantee the Best Sentence

Greedy Search makes the best **local decision** at every timestep.

But our actual objective is to maximize the **global probability of the entire sequence**.

These two objectives are not necessarily equivalent.

Suppose at the first timestep:

$$
P(A\mid x)=0.6
$$

and:

$$
P(B\mid x)=0.4
$$

Greedy Search selects:

$$
A
$$

because:

$$
0.6>0.4
$$

Now suppose at the second timestep:

$$
P(C\mid A,x)=0.1
$$

while:

$$
P(D\mid B,x)=0.9
$$

Then the probability of the two possible sequences is:

$$
P(A,C\mid x)
=
0.6\times0.1
=
0.06
$$

while:

$$
P(B,D\mid x)
=
0.4\times0.9
=
0.36
$$

Therefore:

$$
P(A\mid x)>P(B\mid x)
$$

but:

$$
P(B,D\mid x)>P(A,C\mid x)
$$

So:

$$
\boxed{
\text{Best First Token}
\neq
\text{Necessarily Part of the Best Sentence}
}
$$

This is the central limitation of Greedy Search.

Greedy Search optimizes:

$$
\boxed{
\text{Local decision}
}
$$

while the real objective is:

$$
\boxed{
\text{Global sequence probability}
}
$$

This motivates the need for better search strategies in sequence generation.

---

# 5. Log Probability

Because the probability of a sentence is a product:

$$
P(y\mid x)
=
\prod_t
P
\left(
y^{\langle t\rangle}
\mid
y^{\langle<t\rangle},
x
\right)
$$

the probability can become extremely small for long sequences.

We therefore work with the log probability:

$$
\boxed{
\log P(y\mid x)
=
\sum_{t=1}^{T_y}
\log
P
\left(
y^{\langle t\rangle}
\mid
y^{\langle<t\rangle},
x
\right)
}
$$

because:

$$
\log(ab)=\log a+\log b
$$

Since logarithm is a monotonically increasing function:

$$
\boxed{
\arg\max_yP(y\mid x)
=
\arg\max_y\log P(y\mid x)
}
$$

Thus, we can score candidate sentences using their **cumulative log probability** rather than multiplying many small probabilities.

---

# Final Mental Model

The central problem is:

$$
\boxed{
y^*
=
\arg\max_yP(y\mid x)
}
$$

with:

$$
\boxed{
P(y\mid x)
=
\prod_t
P
\left(
y^{\langle t\rangle}
\mid
y^{\langle<t\rangle},x
\right)
}
$$

The difficulty is that the number of possible output sequences is enormous:

$$
\boxed{
V^{T_y}
}
$$

Greedy Search provides a simple approximation:

$$
\boxed{
\text{Choose the highest-probability token at each timestep}
}
$$

but:

$$
\boxed{
\text{locally optimal choices do not necessarily produce the globally most probable sentence}
}
$$

Therefore, the core idea of **Picking the Most Likely Sentence** is:

$$
\boxed{
\text{Model}
\rightarrow
P(y\mid x)
\rightarrow
\text{Search the sequence space}
\rightarrow
\text{Select a high-probability sentence}
}
$$

The next topic, **Beam Search**, addresses precisely the limitation of Greedy Search without attempting to enumerate the entire sequence space.

# Beam Search

Beam Search is a **decoding/search algorithm** used in sequence generation to find a high-probability output sequence without exhaustively searching the entire sequence space.

The starting objective is:

$$
\boxed{
y^*
=
\arg\max_y P(y\mid x)
}
$$

where:

$$
P(y\mid x)
=
\prod_{t=1}^{T_y}
P\left(
y^{\langle t\rangle}
\mid
y^{\langle<t\rangle},x
\right)
$$

The problem is that the number of possible output sequences grows extremely quickly with sequence length. Beam Search addresses this by keeping several promising candidate sequences instead of keeping only one as in Greedy Search.

---

# 1. Core Idea

Let:

$$
\boxed{
B=\text{beam width}
}
$$

Beam Search keeps the best $B$ **partial sequences** at every timestep.

For example, if:

$$
B=3
$$

then instead of keeping only the best candidate, we maintain:

$$
\boxed{
3\text{ most promising sequences}
}
$$

Suppose at the first timestep:

$$
P(A\mid x)=0.5
$$

$$
P(B\mid x)=0.3
$$

$$
P(C\mid x)=0.15
$$

$$
P(D\mid x)=0.05
$$

With:

$$
B=3
$$

we keep:

$$
A,\quad B,\quad C
$$

and discard:

$$
D
$$

At the next timestep, each retained sequence is expanded with possible next tokens.

Thus:

$$
A\rightarrow A_1,A_2,\ldots
$$

$$
B\rightarrow B_1,B_2,\ldots
$$

$$
C\rightarrow C_1,C_2,\ldots
$$

All resulting candidates are scored, and only the best $B$ are kept again.

Therefore the basic loop is:

$$
\boxed{
\text{Expand}
\rightarrow
\text{Score}
\rightarrow
\text{Keep Top }B
\rightarrow
\text{Repeat}
}
$$

---

# 2. Sequence Score

To compare candidate sequences, we use their cumulative log probability.

For a partial sequence:

$$
y=
(y^{\langle1\rangle},\ldots,y^{\langle t\rangle})
$$

define:

$$
\boxed{
S(y)
=
\log P(y\mid x)
}
$$

Using the chain rule:

$$
\boxed{
S(y)
=
\sum_{k=1}^{t}
\log
P
\left(
y^{\langle k\rangle}
\mid
y^{\langle<k\rangle},x
\right)
}
$$

Suppose the current sequence has score:

$$
S(y_{1:t-1})
$$

and we want to append token $w$.

The new score is:

$$
\boxed{
S(y_{1:t})
=
S(y_{1:t-1})
+
\log
P
\left(
w
\mid
y_{1:t-1},x
\right)
}
$$

Thus every time we append a token, we add its log probability to the sequence score.

Log probability is used because:

$$
P(y\mid x)
=
\prod_t P_t
$$

can become extremely small for long sequences, while:

$$
\log P(y\mid x)
=
\sum_t\log P_t
$$

is easier to compute and compare.

Since logarithm is monotonically increasing:

$$
\boxed{
\arg\max_yP(y\mid x)
=
\arg\max_y\log P(y\mid x)
}
$$

---

# 3. A Small Example with $B=2$

Suppose:

$$
B=2
$$

At timestep 1:

$$
P(A\mid x)=0.6
$$

$$
P(B\mid x)=0.4
$$

so the beam contains:

$$
A,\quad B
$$

At timestep 2, suppose:

$$
P(C\mid A,x)=0.1
$$

$$
P(D\mid A,x)=0.3
$$

$$
P(C\mid B,x)=0.9
$$

$$
P(D\mid B,x)=0.05
$$

We obtain four candidates.

For $(A,C)$:

$$
P(A,C\mid x)
=
0.6\times0.1
=
0.06
$$

For $(A,D)$:

$$
P(A,D\mid x)
=
0.6\times0.3
=
0.18
$$

For $(B,C)$:

$$
P(B,C\mid x)
=
0.4\times0.9
=
0.36
$$

For $(B,D)$:

$$
P(B,D\mid x)
=
0.4\times0.05
=
0.02
$$

Since:

$$
0.36>0.18>0.06>0.02
$$

and:

$$
B=2
$$

we keep:

$$
\boxed{
(B,C)
\quad\text{and}\quad
(A,D)
}
$$

The important point is that `B` was not the best choice at timestep 1:

$$
P(B\mid x)=0.4<0.6=P(A\mid x)
$$

but Beam Search did **not discard it immediately**. Keeping several hypotheses allowed the sequence beginning with `B` to become the best candidate later.

---

# 4. Beam Search vs. Greedy Search

Greedy Search uses:

$$
\boxed{
B=1
}
$$

so only one candidate survives at each timestep.

Therefore:

$$
\boxed{
B=1
\Rightarrow
\text{Greedy Search}
}
$$

Beam Search uses:

$$
\boxed{
B>1
}
$$

and maintains multiple hypotheses.

The conceptual difference is:

$$
\boxed{
\text{Greedy}
=
\text{commit to one local choice}
}
$$

while:

$$
\boxed{
\text{Beam Search}
=
\text{keep several plausible alternatives}
}
$$

This allows Beam Search to avoid some of the problems caused by premature local decisions.

---

# 5. Beam Search Procedure

Start with the special start-of-sequence token:

$$
\langle SOS\rangle
$$

with score:

$$
S=0
$$

At every timestep:

$$
\boxed{
\text{1. Predict next-token probabilities}
}
$$

For every sequence currently in the beam, the decoder produces:

$$
P(w\mid y_{<t},x)
$$

Then:

$$
\boxed{
\text{2. Expand each hypothesis}
}
$$

Each current sequence is extended with possible next tokens.

Next:

$$
\boxed{
\text{3. Compute cumulative scores}
}
$$

using:

$$
S_{\text{new}}
=
S_{\text{old}}
+
\log P(w\mid y_{<t},x)
$$

Then:

$$
\boxed{
\text{4. Keep the top }B\text{ candidates}
}
$$

Finally, repeat until sequences generate:

$$
\langle EOS\rangle
$$

or reach a maximum length.

Thus:

$$
\boxed{
\text{Expand}
\rightarrow
\text{Score}
\rightarrow
\text{Rank}
\rightarrow
\text{Prune}
\rightarrow
\text{Repeat}
}
$$

---

# 6. Beam Width

The beam width controls how many hypotheses are preserved.

If:

$$
B=1
$$

then:

$$
\boxed{
\text{Beam Search becomes Greedy Search}
}
$$

If:

$$
B>1
$$

then multiple candidate sequences survive.

Increasing $B$ generally means:

$$
B\uparrow
\Rightarrow
\text{more hypotheses explored}
$$

but also:

$$
B\uparrow
\Rightarrow
\text{higher computation}
$$

Therefore:

$$
\boxed{
\text{Beam width}
\leftrightarrow
\text{Search quality vs. computational cost}
}
$$

A larger beam does not guarantee the exact globally optimal sequence, because a potentially good sequence can still be pruned at an earlier timestep.

---

# 7. Length Bias

There is an important issue with using raw cumulative log probability.

Because:

$$
0<P(w\mid\cdot)\le1
$$

we have:

$$
\log P(w\mid\cdot)\le0
$$

Therefore, as a sequence becomes longer, more non-positive terms are added:

$$
S(y)
=
\sum_t\log P_t
$$

This can cause Beam Search to favor shorter sequences.

For example:

$$
P(A)=0.8
$$

gives:

$$
\log P(A)=\log0.8
$$

while:

$$
P(B)=0.8\times0.8\times0.8
$$

gives:

$$
\log P(B)=3\log0.8
$$

and since:

$$
\log0.8<0
$$

the longer sequence receives a smaller raw log probability.

This creates a **length bias**.

Practical systems can therefore use techniques such as **length normalization** or **length penalty** when ranking candidate sequences.

At this stage, the important idea is simply:

$$
\boxed{
\text{Raw cumulative log probability can favor shorter sequences}
}
$$

---

# Final Mental Model

The original objective is:

$$
\boxed{
y^*
=
\arg\max_yP(y\mid x)
}
$$

but the number of possible sequences is enormous.

Greedy Search keeps only one hypothesis:

$$
\boxed{
B=1
}
$$

Beam Search keeps the top $B$ partial sequences:

$$
\boxed{
B>1
}
$$

The score of a candidate sequence is:

$$
\boxed{
S(y)
=
\sum_t
\log
P(y_t\mid y_{<t},x)
}
$$

At every timestep:

$$
\boxed{
\text{Expand}
\rightarrow
\text{Score}
\rightarrow
\text{Rank}
\rightarrow
\text{Keep Top }B
}
$$

The central idea is:

$$
\boxed{
\text{Greedy Search commits to one path}
}
$$

while:

$$
\boxed{
\text{Beam Search keeps several promising paths alive}
}
$$

Therefore, Beam Search provides a practical approximation to:

$$
\arg\max_yP(y\mid x)
$$

without having to enumerate the entire sequence space.

# Refinements to Beam Search

Basic Beam Search uses cumulative log probability to rank candidate sequences:

$$
\boxed{
S(y)
=
\sum_{t=1}^{T_y}
\log
P
\left(
y^{\langle t\rangle}
\mid
y^{\langle<t\rangle},x
\right)
}
$$

However, this scoring strategy has some limitations, especially **length bias**. Practical Beam Search can therefore be refined by adjusting the sequence score and handling completed hypotheses properly.

---

# 1. Length Normalization / Length Penalty

Because:

$$
0<P(y_t\mid\cdot)\le1
$$

we have:

$$
\log P(y_t\mid\cdot)\le0
$$

Therefore, adding more tokens usually makes the cumulative log probability smaller:

$$
S(y)
=
\sum_t\log P(y_t\mid y_{<t},x)
$$

As a result, longer sequences tend to receive lower raw scores.

For example:

$$
P(y_1)=0.8
$$

gives:

$$
\log P(y_1)=\log0.8
$$

while a longer sequence:

$$
P(y_2)=0.8\times0.8\times0.8
$$

has:

$$
\log P(y_2)=3\log0.8
$$

Since:

$$
\log0.8<0
$$

we have:

$$
3\log0.8<\log0.8
$$

This creates **length bias**:

$$
\boxed{
\text{Raw cumulative log probability}
\rightarrow
\text{tendency to prefer shorter sequences}
}
$$

A simple way to reduce this bias is to normalize by sequence length:

$$
\boxed{
S_{\text{norm}}(y)
=
\frac{1}{T_y}
\sum_{t=1}^{T_y}
\log
P
\left(
y^{\langle t\rangle}
\mid
y^{\langle<t\rangle},x
\right)
}
$$

Instead of comparing the total log probability, we compare the **average log probability per token**.

More generally, practical systems can use a **length penalty** rather than simple division by $T_y$.

The core idea is:

$$
\boxed{
\text{Adjust the sequence score according to sequence length}
}
$$

so that Beam Search is less biased toward very short outputs.

---

# 2. Handling Finished Hypotheses

Sequence generation usually uses a special token:

$$
\langle EOS\rangle
$$

to indicate that the output sequence has ended.

For example:

$$
\text{I love this movie }\langle EOS\rangle
$$

Once a hypothesis generates:

$$
\langle EOS\rangle
$$

it becomes a **finished hypothesis**.

It should no longer be expanded as an active sequence.

We can therefore distinguish between:

$$
\boxed{
\text{Active Hypotheses}
}
$$

and:

$$
\boxed{
\text{Finished Hypotheses}
}
$$

Active hypotheses continue to be expanded:

$$
\text{Active}
\rightarrow
\text{Expand}
\rightarrow
\text{New Candidates}
$$

while:

$$
\text{Finished}
\rightarrow
\text{Store for Final Ranking}
$$

The search continues until enough hypotheses are completed or a maximum sequence length is reached.

The final output is selected from the completed hypotheses using the appropriate score:

$$
\boxed{
\text{Final Score}
=
\text{Sequence Score}
+
\text{Length Adjustment}
}
$$

---

# 3. Refined Beam Search

A basic Beam Search loop can therefore be extended to:

$$
\boxed{
\text{Predict}
\rightarrow
\text{Expand}
\rightarrow
\text{Score}
\rightarrow
\text{Length Adjustment}
\rightarrow
\text{Prune}
\rightarrow
\text{Repeat}
}
$$

while simultaneously handling completed sequences:

$$
\boxed{
\langle EOS\rangle
\rightarrow
\text{Finished Hypothesis}
}
$$

The two key refinements are:

$$
\boxed{
\text{1. Length Normalization / Length Penalty}
}
$$

to reduce:

$$
\boxed{
\text{length bias}
}
$$

and:

$$
\boxed{
\text{2. Proper Handling of Finished Hypotheses}
}
$$

so that sequences that have generated $\langle EOS\rangle$ are preserved for final ranking instead of being expanded further.

---

# Final Mental Model

$$
\boxed{
\text{Beam Search}
=
\text{Approximate Search}
+
\text{Improved Scoring}
+
\text{Proper Termination}
}
$$

The key ideas are:

$$
\boxed{
\text{Length Adjustment}
\rightarrow
\text{reduce preference for overly short sequences}
}
$$

and:

$$
\boxed{
\langle EOS\rangle
\rightarrow
\text{mark a hypothesis as completed}
}
$$

# Error Analysis in Beam Search

When a sequence-to-sequence model generates an incorrect output, the error can come from either the **model** or the **Beam Search algorithm**.

Suppose:

$$
y^*=\text{correct output}
$$

and:

$$
\hat y=\text{Beam Search output}
$$

The key comparison is:

$$
P(y^*\mid x)
\quad\text{vs.}\quad
P(\hat y\mid x)
$$

## 1. Model Error

If:

$$
\boxed{
P(y^*\mid x)\le P(\hat y\mid x)
}
$$

then the model itself assigns the incorrect output at least as much probability as the correct output.

Therefore:

$$
\boxed{
\text{Model Error}
}
$$

The solution should focus on improving the model, training data, architecture, or objective.

---

## 2. Beam Search Error

If:

$$
\boxed{
P(y^*\mid x)>P(\hat y\mid x)
}
$$

but Beam Search still produces $\hat y$, then the model actually considers the correct output better, but the search procedure failed to find it.

Therefore:

$$
\boxed{
\text{Beam Search Error}
}
$$

The solution should focus on the decoding procedure, such as beam width or scoring strategy.

---

## 3. Practical Error Analysis

For incorrect examples in a development set:

$$
\hat y\neq y^*
$$

compare:

$$
P(y^*\mid x)
\quad\text{and}\quad
P(\hat y\mid x)
$$

Then classify:

| Condition | Error |
|---|---|
| $P(y^*\mid x)>P(\hat y\mid x)$ | Beam Search Error |
| $P(y^*\mid x)\le P(\hat y\mid x)$ | Model Error |

The proportion of each type tells us **where to spend effort**.

If most errors are Model Errors, improve the model.

If many are Beam Search Errors, improve the search strategy.

---

# Final Mental Model

$$
\boxed{
\text{Bad Output}
\rightarrow
\text{Model Error or Beam Search Error}
}
$$

The key diagnostic is:

$$
\boxed{
P(y^*\mid x)>P(\hat y\mid x)
\Rightarrow
\text{Beam Search Error}
}
$$

$$
\boxed{
P(y^*\mid x)\le P(\hat y\mid x)
\Rightarrow
\text{Model Error}
}
$$

The purpose of Error Analysis in Beam Search is therefore to **separate model quality from search quality before deciding what to improve**.

# BLEU Score

**BLEU** (*Bilingual Evaluation Understudy*) is an automatic metric for evaluating machine translation by comparing a model-generated translation with one or more **reference translations**.

The core idea is:

$$
\boxed{
\text{More n-gram overlap with the reference}
\rightarrow
\text{Higher BLEU}
}
$$

BLEU mainly measures **lexical overlap**, rather than directly evaluating semantic equivalence.

---

# 1. Modified n-gram Precision

BLEU evaluates overlap at different n-gram levels:

$$
n=1,2,3,4
$$

corresponding to:

- unigram;
- bigram;
- trigram;
- 4-gram.

For each $n$, BLEU computes a **modified precision**:

$$
\boxed{
p_n
=
\frac{
\sum_{\text{n-grams}}
\text{clipped count}
}{
\sum_{\text{n-grams}}
\text{candidate count}
}
}
$$

The clipping prevents a candidate from receiving excessive credit by repeatedly generating the same word or phrase.

For an n-gram:

$$
\boxed{
\text{clipped count}
=
\min
\left(
\text{count}_{candidate},
\text{count}_{reference}
\right)
}
$$

For example, if the candidate contains `the` 5 times but the reference contains it only once, the candidate receives at most one match for `the`.

---

# 2. Combining Multiple n-gram Precisions

BLEU combines the different n-gram precisions using a **geometric mean**:

$$
\boxed{
\text{BLEU}
=
BP
\cdot
\exp
\left(
\sum_{n=1}^{N}
w_n\log p_n
\right)
}
$$

Typically:

$$
N=4
$$

and:

$$
w_1=w_2=w_3=w_4=\frac14
$$

so:

$$
\boxed{
\text{BLEU}
=
BP
\cdot
(p_1p_2p_3p_4)^{1/4}
}
$$

The geometric mean makes BLEU sensitive to all n-gram levels.

If one $p_n$ is very low, the overall score is reduced substantially.

Thus the model cannot obtain a high BLEU score simply by matching individual words while producing poor multi-word structures.

---

# 3. Brevity Penalty

Precision alone can favor very short candidates.

For example, if the reference is:

> The cat is sitting on the mat

and the candidate is:

> cat

the candidate may achieve reasonable unigram precision despite being incomplete.

BLEU therefore uses a **Brevity Penalty (BP)**.

Let:

$$
c=\text{candidate length}
$$

and:

$$
r=\text{reference length}
$$

Then:

$$
\boxed{
BP=
\begin{cases}
1,&c>r\\
e^{1-r/c},&c\le r
\end{cases}
}
$$

Therefore:

- if the candidate is at least as long as the reference, it receives no brevity penalty;
- if the candidate is shorter, its BLEU score is reduced.

The complete metric is therefore:

$$
\boxed{
\text{BLEU}
=
\text{Brevity Penalty}
\times
\text{Geometric Mean of n-gram Precisions}
}
$$

---

# 4. What BLEU Measures

BLEU primarily measures:

$$
\boxed{
\text{n-gram overlap between candidate and reference}
}
$$

This makes it useful for automatically comparing machine translation systems.

However, BLEU does not directly understand semantic equivalence.

For example:

Reference:

> The cat is sleeping.

Candidate:

> A feline is resting.

The two sentences may convey similar meaning, but their lexical overlap is low.

Therefore:

$$
\boxed{
\text{High BLEU}
\not\Rightarrow
\text{perfect translation}
}
$$

and:

$$
\boxed{
\text{Low BLEU}
\not\Rightarrow
\text{poor semantic quality}
}
$$

BLEU should therefore be treated as an **automatic evaluation metric**, not a complete measure of translation quality.

---

# Final Mental Model

BLEU can be remembered through three components:

$$
\boxed{
\text{BLEU}
=
BP
\times
\text{n-gram overlap score}
}
$$

First:

$$
\boxed{
\text{Candidate}
\rightarrow
\text{n-gram matching}
\rightarrow
p_1,p_2,\ldots,p_N
}
$$

Then:

$$
\boxed{
\text{Geometric Mean}
=
\exp
\left(
\sum_n w_n\log p_n
\right)
}
$$

Finally:

$$
\boxed{
\text{BLEU}
=
BP
\cdot
\exp
\left(
\sum_n w_n\log p_n
\right)
}
$$

The key ideas are:

$$
\boxed{
\text{Modified n-gram Precision}
\rightarrow
\text{measure lexical overlap}
}
$$

$$
\boxed{
\text{Brevity Penalty}
\rightarrow
\text{penalize overly short translations}
}
$$

Therefore:

$$
\boxed{
\text{BLEU}
=
\text{automatic translation evaluation based primarily on n-gram overlap}
}
$$

# Attention Model Intuition

Attention was introduced to address an important limitation of the traditional **encoder-decoder architecture**:

> The encoder must compress the entire input sequence into a single fixed-length representation, and the decoder must use that representation to generate the entire output sequence.

For long sequences, this creates a serious information bottleneck.

Attention replaces this fixed bottleneck with a mechanism that allows the decoder to **dynamically focus on different parts of the input at each decoding timestep**.

The core idea is:

$$
\boxed{
\text{Decoder}
\rightarrow
\text{look at all encoder hidden states}
\rightarrow
\text{focus on relevant information}
\rightarrow
\text{generate next token}
}
$$

---

# 1. The Limitation of the Traditional Encoder-Decoder

Consider a machine translation task:

> The cat is sitting on the mat.

$\rightarrow$

> Le chat est assis sur le tapis.

The encoder processes the input sequence:

$$
x^{\langle1\rangle},
x^{\langle2\rangle},
\ldots,
x^{\langle T_x\rangle}
$$

and produces hidden states:

$$
a^{\langle1\rangle},
a^{\langle2\rangle},
\ldots,
a^{\langle T_x\rangle}
$$

In the traditional encoder-decoder architecture, the decoder is typically initialized from a single final representation:

$$
a^{\langle T_x\rangle}
$$

Conceptually:

$$
\boxed{
\text{Entire Input}
\rightarrow
\text{One Fixed-Length Vector}
\rightarrow
\text{Entire Output}
}
$$

This creates a **fixed-length information bottleneck**.

For a short sentence, this may work reasonably well.

However, as the input becomes longer:

$$
T_x\uparrow
$$

the encoder must compress more information into essentially the same representation.

The decoder then has to recover all the information it needs from that bottleneck.

---

# 2. The Core Idea of Attention

Attention changes this architecture.

Instead of forcing the decoder to rely on one fixed representation, we keep **all encoder hidden states**:

$$
\boxed{
a^{\langle1\rangle},
a^{\langle2\rangle},
\ldots,
a^{\langle T_x\rangle}
}
$$

Then, at every decoder timestep, the decoder asks:

> **Which parts of the input are most relevant for generating the current output token?**

Therefore, every decoder timestep gets its own context vector:

$$
\boxed{
c^{\langle1\rangle},
c^{\langle2\rangle},
\ldots,
c^{\langle T_y\rangle}
}
$$

The architecture becomes:

![Encoder-Decoder Attention](https://lilianweng.github.io/lil-log/assets/images/encoder-decoder-attention.png)

$$
\boxed{
\text{Encoder Hidden States}
\rightarrow
\text{Attention}
\rightarrow
\text{Dynamic Context Vector}
\rightarrow
\text{Decoder}
}
$$

The key change is:

$$
\boxed{
\text{One fixed context vector}
\rightarrow
\text{One context vector per decoder timestep}
}
$$

---

# 3. Intuition Through Machine Translation

Consider:

> The cat is sitting on the mat.

$\rightarrow$

> Le chat est assis sur le tapis.

The decoder does not need exactly the same information at every timestep.

When generating:

> `chat`

the model should mainly focus on:

> `cat`

When generating:

> `assis`

the model should pay more attention to:

> `sitting`

When generating:

> `tapis`

the relevant input is:

> `mat`

Therefore:

$$
\boxed{
\text{Different output tokens}
\rightarrow
\text{different focus on the input}
}
$$

This is the intuition behind Attention.

The model does not permanently compress the entire input into one vector and hope that the decoder remembers everything.

Instead:

$$
\boxed{
\text{Look back at the input whenever necessary}
}
$$

---

# 4. Encoder Hidden States

The encoder processes the input sequence and produces:

$$
a^{\langle1\rangle},
a^{\langle2\rangle},
\ldots,
a^{\langle T_x\rangle}
$$

Each hidden state contains information associated with its corresponding input position together with contextual information from the sequence.

Conceptually:

$$
x^{\langle1\rangle}
\rightarrow
a^{\langle1\rangle}
$$

$$
x^{\langle2\rangle}
\rightarrow
a^{\langle2\rangle}
$$

$$
\vdots
$$

$$
x^{\langle T_x\rangle}
\rightarrow
a^{\langle T_x\rangle}
$$

Traditional encoder-decoder models mainly rely on:

$$
a^{\langle T_x\rangle}
$$

Attention instead keeps access to:

$$
\boxed{
\left\{
a^{\langle1\rangle},
a^{\langle2\rangle},
\ldots,
a^{\langle T_x\rangle}
\right\}
}
$$

throughout decoding.

---

# 5. Attention Scores

Suppose the decoder is currently at timestep $t$.

It has a decoder hidden state:

$$
s^{\langle t-1\rangle}
$$

The decoder compares this state with every encoder hidden state:

$$
a^{\langle1\rangle},
a^{\langle2\rangle},
\ldots,
a^{\langle T_x\rangle}
$$

to determine how relevant each input position is.

For encoder position $i$, we compute an **attention score**:

$$
\boxed{
e^{\langle t,i\rangle}
}
$$

Conceptually:

$$
e^{\langle t,i\rangle}
=
\text{relevance of encoder position }i
\text{ for decoder timestep }t
$$

A larger score means that the model considers that encoder state more relevant for the current decoding step.

At this stage, these are only **raw scores**, not probabilities.

---

# 6. Attention Weights

The scores are normalized using softmax:

$$
\boxed{
\alpha^{\langle t,i\rangle}
=
\frac{
\exp(e^{\langle t,i\rangle})
}{
\sum_{k=1}^{T_x}
\exp(e^{\langle t,k\rangle})
}
}
$$

The resulting values satisfy:

$$
0\le
\alpha^{\langle t,i\rangle}
\le1
$$

and:

$$
\boxed{
\sum_{i=1}^{T_x}
\alpha^{\langle t,i\rangle}
=
1
}
$$

Therefore, $\alpha^{\langle t,i\rangle}$ can be interpreted as:

> **How much attention does the decoder place on encoder position $i$ at timestep $t$?**

For example:

| Input word | Attention weight |
|---|---:|
| The | 0.03 |
| cat | 0.72 |
| is | 0.04 |
| sitting | 0.08 |
| on | 0.03 |
| the | 0.03 |
| mat | 0.07 |

When generating `chat`, the model is therefore focusing strongly on `cat`.

---

# 7. Context Vector

The attention weights are used to compute a weighted sum of the encoder hidden states:

$$
\boxed{
c^{\langle t\rangle}
=
\sum_{i=1}^{T_x}
\alpha^{\langle t,i\rangle}
a^{\langle i\rangle}
}
$$

This is the **context vector** for decoder timestep $t$.

For example, if the model strongly attends to `cat`:

$$
\alpha^{\langle t,\text{cat}\rangle}=0.72
$$

then:

$$
c^{\langle t\rangle}
\approx
0.72a_{\text{cat}}
+
\text{small contributions from other encoder states}
$$

Thus the decoder receives a representation dominated by the information it currently needs.

This is the key mechanism:

$$
\boxed{
\text{Attention Weights}
\rightarrow
\text{Weighted Sum}
\rightarrow
\text{Context Vector}
}
$$

---

# 8. Attention Changes at Every Decoder Timestep

This is one of the most important ideas to understand.

The attention distribution is not fixed.

At decoder timestep $t$:

$$
\alpha^{\langle t\rangle}
=
\left[
\alpha^{\langle t,1\rangle},
\ldots,
\alpha^{\langle t,T_x\rangle}
\right]
$$

At the next timestep:

$$
\alpha^{\langle t+1\rangle}
\neq
\alpha^{\langle t\rangle}
$$

in general.

Therefore:

$$
\boxed{
c^{\langle t\rangle}
\neq
c^{\langle t+1\rangle}
}
$$

The decoder effectively gets a **different view of the input at each timestep**.

For example:

$$
\text{Generate "chat"}
\rightarrow
\text{focus on "cat"}
$$

$$
\text{Generate "assis"}
\rightarrow
\text{focus on "sitting"}
$$

$$
\text{Generate "tapis"}
\rightarrow
\text{focus on "mat"}
$$

Therefore:

$$
\boxed{
\text{Each output token can access a different part of the input}
}
$$

---

# 9. How the Decoder Uses the Context Vector

Once we have:

$$
c^{\langle t\rangle}
$$

the decoder uses it together with its own state to generate the next output.

Conceptually:

$$
\boxed{
s^{\langle t-1\rangle}
+
c^{\langle t\rangle}
\rightarrow
s^{\langle t\rangle}
\rightarrow
y^{\langle t\rangle}
}
$$

The decoder therefore has access to two types of information:

$$
\boxed{
\text{Previous decoding information}
+
\text{Relevant encoder information}
}
$$

The complete flow becomes:

$$
\boxed{
\text{Encoder States}
\rightarrow
\text{Attention Scores}
\rightarrow
\text{Attention Weights}
\rightarrow
\text{Context Vector}
\rightarrow
\text{Decoder}
\rightarrow
\text{Next Token}
}
$$

---

# 10. Why Attention Solves the Bottleneck

Without Attention:

$$
\boxed{
a^{\langle1\rangle},
\ldots,
a^{\langle T_x\rangle}
\rightarrow
\text{one fixed representation}
\rightarrow
\text{decoder}
}
$$

With Attention:

$$
\boxed{
a^{\langle1\rangle},
\ldots,
a^{\langle T_x\rangle}
\rightarrow
\text{dynamic access during decoding}
}
$$

The decoder no longer needs to extract all relevant information from one fixed vector.

Instead, it can repeatedly:

$$
\boxed{
\text{look back}
\rightarrow
\text{focus}
\rightarrow
\text{extract relevant information}
}
$$

Thus:

$$
\boxed{
\text{Fixed-Length Bottleneck}
\rightarrow
\text{Dynamic Access to Encoder States}
}
$$

This is the fundamental reason Attention significantly improves sequence-to-sequence models, especially for long input sequences.

---

# 11. Attention as Soft Selection

Attention can be understood as a **soft selection mechanism**.

The decoder has:

$$
T_x
$$

encoder hidden states.

Rather than choosing exactly one:

$$
a^{\langle i\rangle}
$$

it assigns a weight to every state:

$$
\alpha^{\langle t,1\rangle},
\ldots,
\alpha^{\langle t,T_x\rangle}
$$

and computes:

$$
c^{\langle t\rangle}
=
\sum_i
\alpha^{\langle t,i\rangle}
a^{\langle i\rangle}
$$

Therefore:

$$
\boxed{
\text{Attention}
=
\text{soft weighted selection over encoder states}
}
$$

It is not necessarily:

$$
\text{choose exactly one input word}
$$

Instead, it can distribute attention across multiple positions.

---

# 12. Attention and Information Flow

Attention also creates a shorter information path between encoder representations and decoder outputs.

Without Attention, information from an early input token must pass through many recurrent states before influencing a later output.

With Attention, an encoder state can directly contribute to a decoder context vector:

$$
a^{\langle i\rangle}
\rightarrow
c^{\langle t\rangle}
\rightarrow
\text{decoder output}
$$

This provides:

$$
\boxed{
\text{Shorter Information Path}
\rightarrow
\text{Better Information Flow}
}
$$

During backpropagation, the loss can also propagate through the attention mechanism back toward the relevant encoder states.

This helps sequence models deal with long-range dependencies more effectively.

---

# 13. Attention Is Not Simply "Finding the Corresponding Word"

A common misconception is that Attention always finds one exact input word corresponding to the current output word.

That is not quite correct.

Attention produces:

$$
\alpha^{\langle t,1\rangle},
\ldots,
\alpha^{\langle t,T_x\rangle}
$$

and combines all encoder states:

$$
c^{\langle t\rangle}
=
\sum_i
\alpha^{\langle t,i\rangle}
a^{\langle i\rangle}
$$

Therefore, an output token can depend on **multiple input positions** with different weights.

Attention should therefore be understood primarily as:

$$
\boxed{
\text{learned relevance weighting}
}
$$

rather than simply:

$$
\boxed{
\text{one-to-one word alignment}
}
$$

Although in machine translation, attention weights can often reveal useful alignment-like patterns.

---

# 14. Complete Intuition

At each decoder timestep:

### Step 1 — Compare the decoder state with every encoder state

$$
s^{\langle t-1\rangle}
\quad\text{vs.}\quad
a^{\langle1\rangle},\ldots,a^{\langle T_x\rangle}
$$

to obtain:

$$
e^{\langle t,i\rangle}
$$

### Step 2 — Convert scores into attention weights

$$
\boxed{
\alpha^{\langle t,i\rangle}
=
\frac{
\exp(e^{\langle t,i\rangle})
}{
\sum_k\exp(e^{\langle t,k\rangle})
}
}
$$

### Step 3 — Build the context vector

$$
\boxed{
c^{\langle t\rangle}
=
\sum_i
\alpha^{\langle t,i\rangle}
a^{\langle i\rangle}
}
$$

### Step 4 — Use the relevant context to generate the next token

$$
\boxed{
s^{\langle t-1\rangle}
+
c^{\langle t\rangle}
\rightarrow
y^{\langle t\rangle}
}
$$

So the complete mechanism is:

$$
\boxed{
\text{Encoder States}
\rightarrow
\text{Attention}
\rightarrow
\text{Relevant Context}
\rightarrow
\text{Decoder}
\rightarrow
\text{Output Token}
}
$$

---

# Final Mental Model

The most important idea to remember is:

$$
\boxed{
\text{Attention replaces the single fixed context bottleneck with a dynamic context vector at every decoder timestep}
}
$$

Or more intuitively:

> **At every step of generation, the decoder looks back at the entire input and decides where to focus.**

The mathematical structure is:

$$
\boxed{
\text{Score}
\rightarrow
\text{Softmax Weights}
\rightarrow
\text{Weighted Sum}
\rightarrow
\text{Context Vector}
\rightarrow
\text{Decoder}
}
$$

with:

$$
\boxed{
\alpha^{\langle t,i\rangle}
=
\frac{
\exp(e^{\langle t,i\rangle})
}{
\sum_k\exp(e^{\langle t,k\rangle})
}
}
$$

and:

$$
\boxed{
c^{\langle t\rangle}
=
\sum_i
\alpha^{\langle t,i\rangle}
a^{\langle i\rangle}
}
$$

The fundamental evolution is therefore:

$$
\boxed{
\text{Fixed Context}
\rightarrow
\text{Dynamic Attention}
\rightarrow
\text{Context at Every Decoding Step}
}
$$

# Attention Model

Attention is a mechanism that allows a neural network to **dynamically select and combine relevant information from a set of representations**.

In the original encoder-decoder architecture, the entire input sequence had to be compressed into a single fixed-length representation. Attention removes this bottleneck by allowing the decoder to access the encoder's hidden states directly and assign different importance to them at each decoding step.

The core idea is:

$$
\boxed{
\text{Fixed Context}
\rightarrow
\text{Dynamic Context at Each Decoding Step}
}
$$

More generally:

$$
\boxed{
\text{Relevant Information}
\rightarrow
\text{Attention Weights}
\rightarrow
\text{Weighted Combination}
}
$$

![Attention Model](https://miro.medium.com/v2/resize:fit:2000/format:webp/1*-0sUZK3pcNfw_2emyhqpng.png)

---

# 1. The Limitation of the Traditional Encoder-Decoder

Consider a machine translation problem:

> The cat is sitting on the mat.

$$
\downarrow
$$

> Le chat est assis sur le tapis.

The encoder processes the input sequence:

$$
x^{\langle1\rangle},
x^{\langle2\rangle},
\ldots,
x^{\langle T_x\rangle}
$$

and produces hidden states:

$$
a^{\langle1\rangle},
a^{\langle2\rangle},
\ldots,
a^{\langle T_x\rangle}
$$

In the traditional encoder-decoder architecture, the decoder relies on a fixed-length representation of the input, often associated with the final encoder state:

$$
a^{\langle T_x\rangle}
$$

Conceptually:

$$
\boxed{
\text{Entire Input}
\rightarrow
\text{One Fixed-Length Representation}
\rightarrow
\text{Entire Output}
}
$$

This creates an information bottleneck.

As the input sequence becomes longer:

$$
T_x\uparrow
$$

the encoder must compress increasingly more information into a fixed-size representation.

This becomes especially problematic when information near the beginning of the input is needed much later during decoding.

---

# 2. The Core Intuition of Attention

Attention removes the requirement that the decoder obtain all information from one fixed representation.

Instead, the decoder keeps access to:

$$
\boxed{
a^{\langle1\rangle},
a^{\langle2\rangle},
\ldots,
a^{\langle T_x\rangle}
}
$$

At every decoder timestep $t$, the model determines which encoder states are most relevant for generating the current output token.

Thus, each decoder timestep obtains its own context vector:

$$
\boxed{
c^{\langle1\rangle},
c^{\langle2\rangle},
\ldots,
c^{\langle T_y\rangle}
}
$$

The fundamental change is:

$$
\boxed{
\text{One fixed context vector}
\rightarrow
\text{One dynamic context vector per decoder timestep}
}
$$

For example, during translation:

$$
\text{Generate "chat"}
\rightarrow
\text{focus more on "cat"}
$$

$$
\text{Generate "assis"}
\rightarrow
\text{focus more on "sitting"}
$$

$$
\text{Generate "tapis"}
\rightarrow
\text{focus more on "mat"}
$$

Therefore:

$$
\boxed{
\text{Different output tokens}
\rightarrow
\text{Different attention distributions over the input}
}
$$

---

# 3. From Encoder States to a Context Vector

The encoder produces:

$$
a^{\langle1\rangle},
a^{\langle2\rangle},
\ldots,
a^{\langle T_x\rangle}
$$

At decoder timestep $t$, the model first assigns an **attention score** to every encoder state:

$$
e^{\langle t,1\rangle},
e^{\langle t,2\rangle},
\ldots,
e^{\langle t,T_x\rangle}
$$

The score represents how relevant encoder position $i$ is for the current decoding step.

The scores are converted into attention weights with softmax:

$$
\boxed{
\alpha^{\langle t,i\rangle}
=
\frac{
\exp(e^{\langle t,i\rangle})
}{
\sum_{j=1}^{T_x}
\exp(e^{\langle t,j\rangle})
}
}
$$

Therefore:

$$
0\le
\alpha^{\langle t,i\rangle}
\le1
$$

and:

$$
\boxed{
\sum_{i=1}^{T_x}
\alpha^{\langle t,i\rangle}
=
1
}
$$

The context vector is then computed as a weighted sum:

$$
\boxed{
c^{\langle t\rangle}
=
\sum_{i=1}^{T_x}
\alpha^{\langle t,i\rangle}
a^{\langle i\rangle}
}
$$

This is the central mathematical operation of Attention.

The complete mechanism is:

$$
\boxed{
\text{Encoder States}
\rightarrow
\text{Attention Scores}
\rightarrow
\text{Attention Weights}
\rightarrow
\text{Weighted Sum}
\rightarrow
c^{\langle t\rangle}
}
$$

---

# 4. Why the Weighted Sum Matters

Suppose at one timestep:

$$
\alpha^{\langle t\rangle}
=
[0.05,0.70,0.10,0.05,0.04,0.03,0.03]
$$

The second encoder state receives most of the weight.

Therefore:

$$
c^{\langle t\rangle}
=
0.05a^{\langle1\rangle}
+
0.70a^{\langle2\rangle}
+
0.10a^{\langle3\rangle}
+\cdots
$$

The resulting context vector is therefore dominated by the information at position 2, while still incorporating information from the other positions.

This is why Attention can be understood as a:

$$
\boxed{
\text{Soft Selection Mechanism}
}
$$

It does not necessarily select exactly one input position.

Instead:

$$
\boxed{
\text{All positions can contribute, but with different weights}
}
$$

---

# 5. Attention Changes at Every Decoder Timestep

The attention weights depend on the current decoder state, so they generally change across timesteps.

At timestep $t$:

$$
\alpha^{\langle t,1\rangle},
\ldots,
\alpha^{\langle t,T_x\rangle}
$$

At timestep $t+1$:

$$
\alpha^{\langle t+1,1\rangle},
\ldots,
\alpha^{\langle t+1,T_x\rangle}
$$

In general:

$$
\boxed{
\alpha^{\langle t\rangle}
\neq
\alpha^{\langle t+1\rangle}
}
$$

Therefore:

$$
\boxed{
c^{\langle t\rangle}
\neq
c^{\langle t+1\rangle}
}
$$

The decoder effectively gets a different view of the input at every output timestep.

This is the key reason Attention is much more flexible than a single fixed context representation.

---

# 6. Types of Attention Mechanisms

The term **Attention** describes a general mechanism, but there are several ways to implement or classify it.

These types are not all mutually exclusive. They describe different aspects of the mechanism:

$$
\boxed{
\begin{aligned}
&\text{Soft vs. Hard}
&&\rightarrow
\text{How information is selected}\\
&\text{Self vs. Encoder-Decoder}
&&\rightarrow
\text{Which representations interact}\\
&\text{Additive vs. Dot-Product}
&&\rightarrow
\text{How attention scores are computed}\\
&\text{Single-Head vs. Multi-Head}
&&\rightarrow
\text{How many attention subspaces are used}
\end{aligned}
}
$$

## 6.1 Soft Attention

Soft Attention produces continuous attention weights:

$$
\alpha_i\in[0,1]
$$

using a differentiable normalization such as softmax:

$$
\alpha_i
=
\frac{\exp(e_i)}
{\sum_j\exp(e_j)}
$$

and computes:

$$
\boxed{
c=\sum_i\alpha_i v_i
}
$$

The key property is that **all candidate values can contribute** to the output.

For example:

$$
c=
0.7v_1+0.2v_2+0.1v_3
$$

rather than choosing exactly one value.

### Why is it important?

The entire computation is differentiable:

$$
\text{Scores}
\rightarrow
\text{Softmax}
\rightarrow
\text{Weighted Sum}
$$

Therefore, gradients can flow through the attention mechanism using standard backpropagation.

This makes Soft Attention easy to integrate into neural networks and is the dominant formulation in modern NLP.

The core idea is:

$$
\boxed{
\text{Continuous relevance weighting}
\rightarrow
\text{Differentiable optimization}
}
$$

---

## 6.2 Hard Attention

Hard Attention makes a discrete selection instead of computing a weighted combination of all inputs.

Conceptually:

$$
i^*=\arg\max_i e_i
$$

and the model uses:

$$
c=v_{i^*}
$$

rather than:

$$
c=\sum_i\alpha_i v_i
$$

Thus:

$$
\boxed{
\text{Soft Attention}
\rightarrow
\text{weighted combination}
}
$$

while:

$$
\boxed{
\text{Hard Attention}
\rightarrow
\text{discrete selection}
}
$$

### Main difficulty

A discrete selection such as:

$$
\arg\max_i
$$

is not differentiable in the ordinary sense.

A small change in the scores may cause:

$$
i^*=2
\rightarrow
i^*=3
$$

which prevents straightforward gradient propagation through the selection.

Therefore, Hard Attention typically requires stochastic optimization, sampling-based methods, or reinforcement-learning-style approaches.

Its fundamental trade-off is:

$$
\boxed{
\text{More discrete/selective}
\quad\leftrightarrow\quad
\text{Harder to optimize}
}
$$

---

## 6.3 Self-Attention

Self-Attention is different from the above distinction because it describes **which representations interact**.

Instead of the decoder attending to encoder states, elements of the same sequence attend to one another.

Suppose the input is:

> The animal did not cross the street because it was tired.

When processing `it`, the model may need information from another token such as `animal`.

Self-Attention allows every token to directly interact with other tokens in the same sequence.

For token $i$:

$$
q_i=W_Qx_i
$$

$$
k_j=W_Kx_j
$$

$$
v_j=W_Vx_j
$$

Then the attention score between token $i$ and token $j$ can be computed from:

$$
e_{ij}
=
\operatorname{score}(q_i,k_j)
$$

and:

$$
\alpha_{ij}
=
\operatorname{softmax}_j(e_{ij})
$$

The new representation for token $i$ is:

$$
\boxed{
z_i
=
\sum_j\alpha_{ij}v_j
}
$$

Therefore:

$$
\boxed{
\text{Each token can dynamically incorporate information from other tokens}
}
$$

This provides a direct path between distant positions and is one of the key ideas behind the Transformer architecture.

### Self-Attention vs. Encoder-Decoder Attention

Encoder-decoder Attention:

$$
\boxed{
\text{Decoder}
\rightarrow
\text{Encoder}
}
$$

Self-Attention:

$$
\boxed{
\text{Sequence}
\rightarrow
\text{same sequence}
}
$$

In Self-Attention, $Q$, $K$, and $V$ originate from the same sequence.

---

## 6.4 Multi-Head Attention

A single attention operation produces one type of weighted interaction.

Multi-Head Attention runs multiple attention mechanisms in parallel:

$$
\boxed{
\text{Multiple Attention Heads}
\rightarrow
\text{Multiple Representation Subspaces}
}
$$

For head $h$:

$$
Q_h=XW_Q^{(h)}
$$

$$
K_h=XW_K^{(h)}
$$

$$
V_h=XW_V^{(h)}
$$

Then:

$$
Z_h
=
\operatorname{Attention}(Q_h,K_h,V_h)
$$

The outputs of all heads are concatenated:

$$
\boxed{
Z
=
\operatorname{Concat}
(Z_1,\ldots,Z_H)
}
$$

and projected:

$$
\boxed{
Y=ZW_O
}
$$

### Why multiple heads?

Different heads can learn different patterns or relationships in the data.

For example, different heads may capture patterns related to:

$$
\text{syntactic relationships}
$$

$$
\text{semantic relationships}
$$

$$
\text{long-range dependencies}
$$

or other structures.

However, these interpretations are not hard constraints. A head is not explicitly told:

> "You must learn syntax."

The different patterns emerge from the learned projection matrices.

The key idea is:

$$
\boxed{
\text{Multi-Head Attention}
=
\text{multiple learned attention subspaces}
}
$$

---

## 6.5 Additive Attention

Additive Attention is a method for calculating the compatibility score between a query and a key.

Instead of using a simple dot product, it uses a small feed-forward neural network.

A common formulation is:

$$
\boxed{
e_i
=
v_a^T
\tanh
\left(
W_q q
+
W_k k_i
+
b_a
\right)
}
$$

Here:

- $q$ is the query;
- $k_i$ is the $i$-th key;
- $W_q$ and $W_k$ are learned projection matrices;
- $v_a$ is a learned vector;
- $b_a$ is a bias.

The resulting scores are normalized:

$$
\alpha_i
=
\frac{
\exp(e_i)
}{
\sum_j\exp(e_j)
}
$$

and used to compute:

$$
c
=
\sum_i\alpha_i v_i
$$

The important distinction is that the compatibility function itself is learned:

$$
\boxed{
(q,k_i)
\rightarrow
\text{Feed-Forward Network}
\rightarrow
e_i
}
$$

This gives the model a more parameterized compatibility function than a simple dot product.

Additive Attention is closely associated with **Bahdanau Attention** and was particularly important in early neural machine translation systems.

---

# 7. Additive Attention vs. Dot-Product Attention

The main contrast is how the raw attention score is computed.

### Dot-Product Attention

$$
\boxed{
e_i=q^Tk_i
}
$$

This is simple and computationally efficient.

### Additive Attention

$$
\boxed{
e_i
=
v_a^T
\tanh
\left(
W_qq+W_kk_i+b_a
\right)
}
$$

This introduces learned projections and a nonlinear compatibility function.

Conceptually:

$$
\boxed{
\text{Dot Product}
\rightarrow
\text{simple similarity}
}
$$

while:

$$
\boxed{
\text{Additive Attention}
\rightarrow
\text{learned compatibility function}
}
$$

Transformer models primarily use scaled dot-product attention because it maps naturally to efficient matrix operations and GPU/TPU computation.

---

# 8. Attention and Information Flow

Attention does more than improve the representation itself. It changes the path through which information flows.

Without Attention, information from an early encoder state must pass through many recurrent transitions before influencing a later decoder output.

With Attention:

$$
a^{\langle i\rangle}
\rightarrow
c^{\langle t\rangle}
\rightarrow
\text{Decoder}
\rightarrow
y^{\langle t\rangle}
$$

An encoder state can therefore contribute much more directly to a decoder output.

Conceptually:

$$
\boxed{
\text{Shorter Information Path}
\rightarrow
\text{Better Information Flow}
}
$$

The backward pass also benefits from these direct connections:

$$
\text{Loss}
\rightarrow
\text{Attention}
\rightarrow
\text{Relevant Encoder States}
$$

This helps sequence models handle long-range dependencies more effectively.

---

# 9. Attention Is Not Necessarily One-to-One Alignment

A common misconception is:

> Attention simply identifies which input word corresponds to the current output word.

That is too restrictive.

Attention produces a distribution:

$$
\alpha^{\langle t,1\rangle},
\ldots,
\alpha^{\langle t,T_x\rangle}
$$

and constructs:

$$
c^{\langle t\rangle}
=
\sum_i
\alpha^{\langle t,i\rangle}
a^{\langle i\rangle}
$$

Therefore, an output token can depend on **multiple input positions simultaneously**.

The most accurate intuition is:

$$
\boxed{
\text{Attention}
=
\text{learned relevance weighting}
}
$$

not simply:

$$
\boxed{
\text{one output token}
\leftrightarrow
\text{one input token}
}
$$

In machine translation, attention weights can nevertheless reveal useful alignment-like patterns.

---

# 10. From Attention Models to Transformers

The original Attention mechanism was introduced as a component inside recurrent encoder-decoder models:

$$
\text{RNN/LSTM Encoder}
\rightarrow
\text{Attention}
\rightarrow
\text{RNN/LSTM Decoder}
$$

The key insight was that a model can dynamically access relevant representations instead of relying on one fixed context vector.

This leads naturally to:

$$
\boxed{
\text{Attention}
\rightarrow
\text{Self-Attention}
\rightarrow
\text{Multi-Head Self-Attention}
\rightarrow
\text{Transformer}
}
$$

The Transformer takes this idea much further by making self-attention the central mechanism for modeling interactions between tokens.

---

# Final Mental Model

The most important abstraction is:

$$
\boxed{
\text{Representations}
\rightarrow
\text{Relevance Scores}
\rightarrow
\text{Attention Weights}
\rightarrow
\text{Weighted Combination}
}
$$

Mathematically:

$$
\boxed{
\alpha_i
=
\frac{
\exp(\operatorname{score}(q,k_i))
}{
\sum_j
\exp(\operatorname{score}(q,k_j))
}
}
$$

and:

$$
\boxed{
c
=
\sum_i\alpha_i v_i
}
$$

For the original encoder-decoder Attention:

$$
\boxed{
\text{Decoder State}
\rightarrow
\text{attend to Encoder States}
\rightarrow
\text{Context Vector}
\rightarrow
\text{Next Output}
}
$$

The major attention types can be organized by what aspect they change:

$$
\boxed{
\begin{aligned}
\text{Soft vs. Hard}
&\rightarrow
\text{How information is selected}\\
\text{Self vs. Encoder-Decoder}
&\rightarrow
\text{Which representations interact}\\
\text{Additive vs. Dot-Product}
&\rightarrow
\text{How scores are computed}\\
\text{Single vs. Multi-Head}
&\rightarrow
\text{How many attention subspaces are used}
\end{aligned}
}
$$

The deepest idea to retain is:

$$
\boxed{
\text{Attention allows a model to dynamically decide which information matters for the current computation.}
}
$$

And the key architectural evolution is:

$$
\boxed{
\text{Fixed-Length Context}
\rightarrow
\text{Dynamic Attention}
\rightarrow
\text{Self-Attention}
\rightarrow
\text{Multi-Head Attention}
\rightarrow
\text{Transformer}
}
$$